# 03 - Data Analysis

Objectif : creer des KPI simples et identifier les batiments prioritaires.

In [8]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

cleaned_path = project_root / 'data' / 'cleaned' / 'buildings_clean.csv'

## Charger les donnees nettoyees

In [9]:
df = pd.read_csv(cleaned_path)
print(f'Lignes : {df.shape[0]:,}')
df.head()

Lignes : 33,268


,building_id,data_year,building_name,building_type,neighborhood,year_built,number_of_floors,property_gfa_total,energy_star_score,site_eui,site_energy_use,electricity_kwh,natural_gas_therms,total_ghg_emissions,ghg_emissions_intensity,compliance_status,energy_star_score_was_missing,natural_gas_was_missing
0,1,2024,Mayflower Park Hotel,Nonresidential,Downtown,1927,12,88434,59.0,61.700001,7113958,1045040,15986,263.3,2.98,Not Compliant,0,0
1,2,2024,Paramount Hotel,Nonresidential,Downtown,1996,11,103566,85.0,71.500000,6330664,787838,36426,208.6,2.36,Compliant,0,0
2,3,2024,Westin Hotel (Parent Building),Nonresidential,Downtown,1969,41,956110,71.0,81.699997,67613264,11279080,58726,2418.2,3.18,Compliant,0,0
3,5,2024,Hotel Max,Nonresidential,Downtown,1926,10,61320,50.0,86.000000,5273981,796976,11648,190.1,3.10,Compliant,0,0
4,8,2024,Warwick Seattle Hotel,Nonresidential,Downtown,1980,18,175580,87.0,97.099998,11983959,1349025,73811,417.9,3.68,Compliant,0,0


## KPI globaux

In [10]:
kpi = pd.DataFrame({
    'metric': [
        'Nombre de batiments',
        'Consommation totale',
        'Site EUI moyen',
        'Emissions CO2 totales',
        'Score ENERGY STAR moyen'
    ],
    'value': [
        len(df),
        round(df['site_energy_use'].sum(), 2),
        round(df['site_eui'].mean(), 2),
        round(df['total_ghg_emissions'].sum(), 2),
        round(df['energy_star_score'].mean(), 2)
    ]
})
kpi

,metric,value
0,Nombre de batiments,3.326800e+04
1,Consommation totale,1.881309e+11
2,Site EUI moyen,5.564000e+01
3,Emissions CO2 totales,5.093195e+06
4,Score ENERGY STAR moyen,7.416000e+01


### Interpretation

Le dataset nettoye contient 33 268 lignes de reporting entre 2015 et 2024. Le score ENERGY STAR moyen est de 74,16, ce qui indique une performance globale plutot correcte. Le Site EUI moyen est de 55,64 : il sert a comparer les batiments en tenant compte de leur surface.

Les emissions totales restent elevees, donc l'analyse doit surtout identifier les batiments et les categories qui concentrent le plus d'impact.

## Analyse par type de batiment

In [11]:
by_type = (
    df.groupby('building_type', as_index=False)
    .agg(
        number_of_buildings=('building_id', 'count'),
        average_site_eui=('site_eui', 'mean'),
        total_energy_use=('site_energy_use', 'sum'),
        total_ghg_emissions=('total_ghg_emissions', 'sum')
    )
    .round(2)
    .sort_values('total_energy_use', ascending=False)
)
by_type.head(10)

,building_type,number_of_buildings,average_site_eui,total_energy_use,total_ghg_emissions
4,Nonresidential,13033,77.53,104566212430,2904339.9
0,Campus,412,109.31,23229952285,746758.5
3,Multifamily Mr (5-9),6718,38.16,20085587406,444162.2
2,Multifamily Lr (1-4),10129,35.49,14400234397,278830.8
1,Multifamily Hr (10+),1229,49.02,12364999784,327415.3
6,Nonresidential Wa,190,126.24,6138191237,200062.4
5,Nonresidential Cos,641,95.61,3916643703,93671.9
7,Sps-District K-12,916,37.54,3429051785,97954.1


### Interpretation

Les batiments non residentiels sont les plus importants dans le total de consommation et d'emissions. C'est logique car ils sont nombreux dans le dataset et leur Site EUI moyen est plus eleve que celui des logements multifamiliaux.

Les campus ont le Site EUI moyen le plus eleve parmi les grandes categories. Cela peut s'expliquer par des usages plus intensifs : equipements, horaires longs, activites techniques ou medicales.

## Top 10 des batiments les plus consommateurs

In [12]:
top_energy = (
    df[['building_name', 'building_type', 'neighborhood', 'site_energy_use', 'total_ghg_emissions']]
    .sort_values('site_energy_use', ascending=False)
    .head(10)
)
top_energy

,building_name,building_type,neighborhood,site_energy_use,total_ghg_emissions
14007,Medical Dental Building,Nonresidential,Downtown,5880242688,475355.3
17474,Medical Dental Building,Nonresidential,Downtown,5859773440,473565.5
5444,Sea-Mar Community Care Center (Nursing Home),Nonresidential,Greater Duwamish,2394201856,126805.1
20639,Seattle Convention Center,Nonresidential,Downtown,639831488,5200.4
7170,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing...,Campus,Greater Duwamish,579555072,23786.0
23720,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing...,Campus,Greater Duwamish,579302208,23037.8
20390,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing...,Campus,Greater Duwamish,567845632,22288.5
16991,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing...,Campus,Greater Duwamish,549317632,21499.9
3645,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing...,Campus,Greater Duwamish,518865440,20400.1
10425,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing...,Campus,Greater Duwamish,518003488,20644.5


### Interpretation

Le top 10 met en avant les batiments qui consomment le plus en valeur absolue. Ces batiments sont prioritaires pour une verification, mais une forte consommation peut aussi venir d'une grande surface ou d'un usage tres intensif.

Les valeurs tres elevees doivent etre lues comme des points d'attention : il faut verifier si elles correspondent a une vraie situation metier ou a une anomalie de reporting.

## Analyse par quartier

In [13]:
by_neighborhood = (
    df.groupby('neighborhood', as_index=False)
    .agg(
        number_of_buildings=('building_id', 'count'),
        average_site_eui=('site_eui', 'mean'),
        total_ghg_emissions=('total_ghg_emissions', 'sum')
    )
    .round(2)
    .sort_values('total_ghg_emissions', ascending=False)
)
by_neighborhood.head(10)

,neighborhood,number_of_buildings,average_site_eui,total_ghg_emissions
3,Downtown,5591,67.54,1991566.5
4,East,4483,52.94,745751.6
5,Greater Duwamish,3356,66.66,570424.7
9,Northeast,2791,56.76,381388.8
6,Lake Union,2762,60.19,370180.2
7,Magnolia / Queen Anne,4273,46.13,281738.9
10,Northwest,2503,47.87,197459.4
1,Central,1338,49.53,123885.9
12,Southwest,1662,46.74,107304.5
0,Ballard,1334,50.24,103159.1


### Interpretation

Downtown est le quartier qui concentre le plus d'emissions de CO2. C'est coherent avec le nombre eleve de batiments et la presence probable de grands immeubles non residentiels.

Greater Duwamish, East et Lake Union ressortent aussi comme zones importantes. Pour un dashboard, ces quartiers meritent un filtre ou une vue dediee.

## Evolution par annee

In [14]:
by_year = (
    df.groupby('data_year', as_index=False)
    .agg(
        number_of_buildings=('building_id', 'count'),
        total_energy_use=('site_energy_use', 'sum'),
        average_site_eui=('site_eui', 'mean'),
        total_ghg_emissions=('total_ghg_emissions', 'sum')
    )
    .round(2)
)
by_year

,data_year,number_of_buildings,total_energy_use,average_site_eui,total_ghg_emissions
0,2015,3130,15273378264,53.42,326353.8
1,2016,3193,16256250755,53.99,357781.2
2,2017,3257,17533759274,55.95,410136.6
3,2018,3330,18032437011,53.49,401163.5
4,2019,3399,23761717774,58.24,885515.3
5,2020,3460,23040936340,55.51,859341.8
6,2021,3103,16163622284,52.29,398193.3
7,2022,3256,18163919189,55.47,445548.2
8,2023,3526,20971533182,64.53,557241.1
9,2024,3614,18933318954,52.77,451920.3


### Interpretation

La consommation et les emissions augmentent fortement en 2019 et 2020, puis baissent en 2021. Le Site EUI moyen atteint son niveau le plus haut en 2023 avec 64,53, avant de redescendre en 2024.

Comme le nombre de batiments reportes change chaque annee, il ne faut pas conclure trop vite a une amelioration ou une degradation. Une partie des variations peut venir du perimetre de reporting.

## Batiments prioritaires

In [15]:
priority_data = df.copy()
priority_data.loc[:, 'high_energy'] = priority_data['site_energy_use'].rank(pct=True)
priority_data.loc[:, 'high_emissions'] = priority_data['total_ghg_emissions'].rank(pct=True)
priority_data.loc[:, 'low_energy_star'] = (100 - priority_data['energy_star_score']).rank(pct=True)

priority_data.loc[:, 'priority_score'] = (
    priority_data['high_energy'] * 0.4
    + priority_data['high_emissions'] * 0.4
    + priority_data['low_energy_star'] * 0.2
)

priority_buildings = (
    priority_data[[
        'building_name', 'building_type', 'neighborhood', 'data_year',
        'site_energy_use', 'total_ghg_emissions', 'energy_star_score',
        'priority_score'
    ]]
    .sort_values('priority_score', ascending=False)
    .head(10)
)

priority_buildings.round(2)

,building_name,building_type,neighborhood,data_year,site_energy_use,total_ghg_emissions,energy_star_score,priority_score
14007,Medical Dental Building,Nonresidential,Downtown,2020,5880242688,475355.3,1.0,1.00
17474,Medical Dental Building,Nonresidential,Downtown,2019,5859773440,473565.5,1.0,1.00
16179,West Seattle Thriftway,Nonresidential,Southwest,2020,141676144,7250.3,1.0,1.00
9775,The Kenney,Multifamily Lr (1-4),Southwest,2022,84370376,4247.8,1.0,1.00
26977,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing...,Campus,Greater Duwamish,2016,448385312,16661.9,8.0,1.00
27055,Seattle Children'S Hospital (Main Campus),Campus,Northeast,2016,253832464,7994.9,10.0,0.99
32812,Four Seasons Hotel & 99 Union Street Private R...,Multifamily Hr (10+),Downtown,2015,57918496,2378.0,1.0,0.99
464,Harborview Medical Center - Ninth & Jefferson ...,Nonresidential,East,2024,71376472,2439.8,4.0,0.99
23800,Seattle Children'S Hospital (Main Campus),Campus,Northeast,2017,258019328,8427.9,13.0,0.99
30247,Seattle Children'S Hospital (Main Campus),Campus,Northeast,2015,251191824,7897.3,13.0,0.99


### Interpretation

Le `priority_score` est un score de priorite entre 0 et 1. Ce n'est pas une note positive : plus le score est proche de 1, plus le batiment doit etre analyse en priorite.

Le score combine trois elements : 40 % pour la consommation d'energie, 40 % pour les emissions de CO2 et 20 % pour un score ENERGY STAR faible. Donc un batiment peut etre prioritaire s'il consomme beaucoup, s'il emet beaucoup, ou si son score ENERGY STAR est faible.

Lecture simple du score :
- proche de 1 : priorite forte, a verifier en premier ;
- autour de 0,5 : priorite moyenne ;
- proche de 0 : priorite faible dans ce dataset.

Dans les resultats, les premiers batiments ont un score proche de 1. Ils combinent souvent forte consommation, fortes emissions et mauvais score ENERGY STAR. Ce sont donc de bons candidats pour une analyse plus detaillee, mais il faut ensuite verifier leur surface, leur usage et la qualite de la donnee.

## Qualite des donnees utilisees

In [16]:
data_quality = pd.DataFrame({
    'metric': [
        'Scores ENERGY STAR completes',
        'Valeurs gaz naturel completees',
        'Part ENERGY STAR completee (%)',
        'Part gaz naturel completee (%)'
    ],
    'value': [
        int(df['energy_star_score_was_missing'].sum()),
        int(df['natural_gas_was_missing'].sum()),
        round(df['energy_star_score_was_missing'].mean() * 100, 2),
        round(df['natural_gas_was_missing'].mean() * 100, 2)
    ]
})
data_quality

,metric,value
0,Scores ENERGY STAR completes,7928.00
1,Valeurs gaz naturel completees,12154.00
2,Part ENERGY STAR completee (%),23.83
3,Part gaz naturel completee (%),36.53


### Interpretation

Environ 23,83 % des scores ENERGY STAR ont ete completes avec la mediane, et 36,53 % des valeurs de gaz naturel ont ete completees avec 0. Cela ne bloque pas l'analyse globale, mais il faut rester prudent sur les conclusions liees a ces deux variables.

Ce point renforce la transparence de l'analyse : les donnees completees ne sont pas cachees, elles restent visibles avec des colonnes indicatrices.